In [1]:
import numpy as np
import pandas as pd

In [2]:
sales = pd.read_csv("C:/Users/shrut/Desktop/walmart/walmart_m5_melted_data.csv")


In [3]:
df_weekly_final=pd.read_csv("C:/Users/shrut/Desktop/walmart/predictions.csv")

In [4]:
# 1. Original sales table se item_id aur dept_id ka unique rishta (mapping) nikalon
dept_mapping = sales[['item_id', 'dept_id']].drop_duplicates()

# 2. Ise apne weekly dataframe mein item_id ke upar merge (join) kar do
df_weekly_final = df_weekly_final.merge(dept_mapping, on='item_id', how='left')

In [5]:
df_test = df_weekly_final[df_weekly_final['predicted_sales'].notna()].copy()

# Har Store aur Item ke liye 4 hafton ki predicted sales ka average nikal kar 52 se multiply karenge
df_demand = df_test.groupby(['store_id', 'item_id']).agg(
    avg_weekly_pred=('predicted_sales', 'mean'),
    sell_price=('sell_price', 'mean'),  # 4 hafton ka average price (usually constant)
    dept_id=('dept_id', 'first')        # Encoded numeric dept_id (0, 1, 2...6)
).reset_index()

# Final Annual Demand formula
df_demand['annual_demand'] = df_demand['avg_weekly_pred'] * 52


conditions = [
    df_demand['dept_id'].isin([0, 1]),   # FOODS_1 & FOODS_2 (Fresh/Frozen)
    df_demand['dept_id'] == 2,           # FOODS_3 (Groceries)
    df_demand['dept_id'] == 6,           # HOUSEHOLD_2 (Large Home Goods)
    df_demand['dept_id'].isin([3, 4, 5])  # HOBBIES_1, HOBBIES_2, HOUSEHOLD_1
]

# 1. Ordering Cost (S) allocation based on truck nature
ordering_costs = [80, 60, 50, 30]
df_demand['ordering_cost'] = np.select(conditions, ordering_costs, default=40)

# 2. Annual Holding Cost (H) = 20% of the real selling price
df_demand['holding_cost_annual'] = df_demand['sell_price'] * 0.20




df_demand['eoq'] = np.sqrt(
    (2 * df_demand['annual_demand'] * df_demand['ordering_cost']) / 
    np.maximum(df_demand['holding_cost_annual'], 0.01)
)

# Float values ko standard order quantities (integers) mein convert kar dete hain
df_demand['eoq'] = np.ceil(df_demand['eoq']).astype(int)


master_catalog_df = df_demand[[
    'store_id', 'item_id', 'dept_id', 'sell_price', 
    'annual_demand', 'holding_cost_annual', 'ordering_cost', 
    'eoq'
]].copy()

# Pehle 5 rows check karne ke liye
print(master_catalog_df.head())


   store_id  item_id  dept_id  sell_price  annual_demand  holding_cost_annual  \
0         0        0        0        2.24     281.487429                0.448   
1         0        1        0        9.48     161.768304                1.896   
2         0        2        0        3.23     307.123303                0.646   
3         0        3        0        1.96     719.260141                0.392   
4         0        4        0        3.54     510.156136                0.708   

   ordering_cost  eoq  
0             80  318  
1             80  117  
2             80  276  
3             80  542  
4             80  340  


In [6]:
output='C:/Users/shrut/Desktop/walmart/master_catalogue.csv'
master_catalog_df.to_csv(output,index=False)

In [7]:
import pandas as pd

dim_product = df_weekly_final[['item_id', 'dept_id']].drop_duplicates().reset_index(drop=True)

dept_mapping = {
    0: "FOODS_1",
    1: "FOODS_2",
    2: "FOODS_3",
    3: "HOBBIES_1",
    4: "HOBBIES_2",
    5: "HOUSEHOLD_1",
    6: "HOUSEHOLD_2"
}

dim_product['broad_dept_name'] = dim_product['dept_id'].map(dept_mapping)
dim_product['broad_dept_num'] = dim_product['dept_id']

dim_product = dim_product.sort_values(by=['broad_dept_num', 'item_id']).reset_index(drop=True)
dim_product['item_sequence'] = dim_product.groupby('broad_dept_num').cumcount() + 1

dim_product['item_name_displayed'] = (
    dim_product['broad_dept_name'].astype(str) + 
    "_item_" + 
    dim_product['item_sequence'].astype(str)
)

dim_product = dim_product[[
    'item_id', 
    'broad_dept_num', 
    'broad_dept_name', 
    'item_name_displayed'
]]

dim_product.to_csv('dim_product_catalog.csv', index=False)

print(dim_product.head(15))

    item_id  broad_dept_num broad_dept_name item_name_displayed
0         0               0         FOODS_1      FOODS_1_item_1
1         1               0         FOODS_1      FOODS_1_item_2
2         2               0         FOODS_1      FOODS_1_item_3
3         3               0         FOODS_1      FOODS_1_item_4
4         4               0         FOODS_1      FOODS_1_item_5
5         5               0         FOODS_1      FOODS_1_item_6
6         6               0         FOODS_1      FOODS_1_item_7
7         7               0         FOODS_1      FOODS_1_item_8
8         8               0         FOODS_1      FOODS_1_item_9
9         9               0         FOODS_1     FOODS_1_item_10
10       10               0         FOODS_1     FOODS_1_item_11
11       11               0         FOODS_1     FOODS_1_item_12
12       12               0         FOODS_1     FOODS_1_item_13
13       13               0         FOODS_1     FOODS_1_item_14
14       14               0         FOOD